# Kalman Filter Sensor Fusion — Interactive Demo

This notebook walks through:
1. The KalmanFilterCV implementation on a synthetic trajectory
2. The One Euro Filter on the same data
3. Side-by-side comparison with visualisations
4. The full benchmark across three noise scenarios

> **Run from the project root:** `jupyter notebook notebooks/demo.ipynb`

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#16213e'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#aaaaaa'
plt.rcParams['xtick.color'] = '#aaaaaa'
plt.rcParams['ytick.color'] = '#aaaaaa'
plt.rcParams['axes.edgecolor'] = '#444466'
plt.rcParams['grid.color'] = '#333355'
plt.rcParams['figure.dpi'] = 120

from kalman_filter import KalmanFilterCV
from one_euro_filter import OneEuroFilter2D
from evaluate import generate_synthetic_trajectory, run_kalman_filter, run_one_euro_filter
from evaluate import compute_rmse, compute_jitter, compute_lag

print('All imports successful!')

## Part 1: Synthetic Trajectory Generation

We generate a constant-velocity ground truth and add Gaussian noise to simulate detector measurements.

In [ ]:
rng = np.random.default_rng(seed=42)
data = generate_synthetic_trajectory(
    n_frames=200,
    velocity=(3.0, 1.5),
    start=(30.0, 30.0),
    measurement_noise_std=10.0,
    drop_fraction=0.0,
    rng=rng,
)
gt   = data['ground_truth']
meas = data['measurements']
mask = data['observed_mask']

print(f'Generated {len(gt)} frames')
print(f'Ground truth range: x=[{gt[:,0].min():.0f},{gt[:,0].max():.0f}], y=[{gt[:,1].min():.0f},{gt[:,1].max():.0f}]')

## Part 2: Running Both Filters

In [ ]:
kf_est  = run_kalman_filter(meas, mask, process_noise_std=2.0, measurement_noise_std=10.0)
oef_est = run_one_euro_filter(meas, mask, min_cutoff=0.5, beta=0.007)

print(f'KF  RMSE: {compute_rmse(kf_est, gt):.2f} px')
print(f'OEF RMSE: {compute_rmse(oef_est, gt):.2f} px')
print()
print(f'KF  Jitter: {compute_jitter(kf_est):.4f} px²')
print(f'OEF Jitter: {compute_jitter(oef_est):.4f} px²')
print()
print(f'KF  Lag: {compute_lag(kf_est, gt):.1f} frames')
print(f'OEF Lag: {compute_lag(oef_est, gt):.1f} frames')

## Part 3: Trajectory Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, title in zip(axes, ['2D Trajectory', 'X Coordinate Over Time']):
    ax.set_facecolor('#16213e')

# Left: 2D trajectory
ax = axes[0]
ax.plot(gt[:,0], gt[:,1], color='#e2e2e2', linewidth=2.5, label='Ground Truth', zorder=3)
ax.scatter(meas[:,0], meas[:,1], color='#f7971e', s=8, alpha=0.5, label='Noisy Measurements', zorder=2)
ax.plot(kf_est[:,0], kf_est[:,1], color='#2ed573', linewidth=2, label='Kalman Filter', zorder=4)
ax.plot(oef_est[:,0], oef_est[:,1], color='#ff6b81', linewidth=2, linestyle='--', label='One Euro Filter', zorder=4)
ax.set_title('2D Trajectory Comparison', color='white', fontweight='bold')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
ax.legend(facecolor='#1a1a2e', edgecolor='#444466', labelcolor='white')

# Right: X over time
ax = axes[1]
t = np.arange(len(gt))
ax.plot(t, gt[:,0], color='#e2e2e2', linewidth=2, label='Ground Truth', zorder=3)
ax.scatter(t, meas[:,0], color='#f7971e', s=6, alpha=0.4, label='Noisy Measurements', zorder=2)
ax.plot(t, kf_est[:,0], color='#2ed573', linewidth=1.5, label='Kalman Filter', zorder=4)
ax.plot(t, oef_est[:,0], color='#ff6b81', linewidth=1.5, linestyle='--', label='One Euro Filter', zorder=4)
ax.set_title('X Position Over Time', color='white', fontweight='bold')
ax.set_xlabel('Frame')
ax.set_ylabel('X (pixels)')
ax.legend(facecolor='#1a1a2e', edgecolor='#444466', labelcolor='white')

fig.patch.set_facecolor('#1a1a2e')
plt.tight_layout()
plt.show()

## Part 4: Kalman Filter — Covariance Uncertainty Ellipses

One key advantage of the KF: it maintains explicit uncertainty estimates (covariance ellipses).
Watch how the ellipses shrink as the filter gains confidence!

In [ ]:
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

def plot_confidence_ellipse(ax, mean, cov_2x2, n_std=2.0, **kwargs):
    """Plot a covariance confidence ellipse."""
    vals, vecs = np.linalg.eigh(cov_2x2)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(np.abs(vals))
    ell = Ellipse(mean, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)

# Run KF step-by-step collecting covariances
kf = KalmanFilterCV(process_noise_std=2.0, measurement_noise_std=10.0)
kf.initialise(meas[0])

positions, covs = [], []
for k in range(1, len(meas)):
    kf.predict()
    kf.update(meas[k])
    positions.append(kf.get_position().copy())
    covs.append(kf.get_covariance()[:2, :2].copy())

positions = np.array(positions)

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_facecolor('#16213e')
fig.patch.set_facecolor('#1a1a2e')

ax.plot(gt[1:,0], gt[1:,1], color='#e2e2e2', linewidth=2, label='Ground Truth', zorder=3)
ax.plot(positions[:,0], positions[:,1], color='#2ed573', linewidth=1.5, label='KF estimate', zorder=4)

# Draw ellipses at every 20th frame
for i in range(0, len(positions), 20):
    plot_confidence_ellipse(
        ax, positions[i], covs[i],
        n_std=2.0,
        facecolor='none', edgecolor='#2ed573', alpha=0.6, linewidth=1.5, linestyle='-'
    )

ax.set_title('KF Trajectory with 2σ Uncertainty Ellipses', color='white', fontweight='bold')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
ax.legend(facecolor='#1a1a2e', edgecolor='#444466', labelcolor='white')
plt.tight_layout()
plt.show()

## Part 5: Robustness to Missed Detections

Simulate 30% frame drops and compare how KF vs OEF handle gaps in observations.

In [ ]:
rng2 = np.random.default_rng(seed=99)
data_drop = generate_synthetic_trajectory(
    n_frames=200,
    measurement_noise_std=8.0,
    drop_fraction=0.30,
    rng=rng2,
)

gt_d   = data_drop['ground_truth']
meas_d = data_drop['measurements']
mask_d = data_drop['observed_mask']

kf_est_d  = run_kalman_filter(meas_d, mask_d, process_noise_std=2.0, measurement_noise_std=8.0)
oef_est_d = run_one_euro_filter(meas_d, mask_d)

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_facecolor('#16213e')
fig.patch.set_facecolor('#1a1a2e')

t = np.arange(len(gt_d))
ax.plot(t, gt_d[:,0], color='#e2e2e2', linewidth=2, label='Ground Truth X', zorder=3)

# Show observed measurements
obs_t = t[mask_d]
ax.scatter(obs_t, meas_d[mask_d, 0], color='#f7971e', s=12, alpha=0.7, label='Observed (70%)', zorder=2)

# Shade dropped frames
dropped = ~mask_d
in_drop = False
drop_start = 0
for i in range(len(dropped)):
    if dropped[i] and not in_drop:
        drop_start = i
        in_drop = True
    elif not dropped[i] and in_drop:
        ax.axvspan(drop_start, i, alpha=0.12, color='#ff4757')
        in_drop = False

ax.plot(t, kf_est_d[:,0], color='#2ed573', linewidth=1.8, label='Kalman Filter', zorder=4)
ax.plot(t, oef_est_d[:,0], color='#ff6b81', linewidth=1.8, linestyle='--', label='One Euro Filter', zorder=4)

ax.set_title('X Position with 30% Frame Drops (red shading = dropped frames)', color='white', fontweight='bold')
ax.set_xlabel('Frame')
ax.set_ylabel('X Position (pixels)')
ax.legend(facecolor='#1a1a2e', edgecolor='#444466', labelcolor='white')
plt.tight_layout()
plt.show()

print(f'KF  RMSE with drops: {compute_rmse(kf_est_d, gt_d):.2f} px')
print(f'OEF RMSE with drops: {compute_rmse(oef_est_d, gt_d):.2f} px')

## Part 6: Full Benchmark

Run the complete benchmark and save all plots to `results/`.

In [ ]:
from evaluate import run_full_benchmark
results = run_full_benchmark(n_frames=300, verbose=True)

In [ ]:
from pathlib import Path
from IPython.display import Image, display

results_dir = Path('../results')
for plot_file in sorted(results_dir.glob('*.png')):
    print(f'\n--- {plot_file.name} ---')
    display(Image(str(plot_file), width=900))